# AI-Powered Influencer Recommendation System & Buyer Intent Estimator

**Guided project notebook**

This notebook explores an influencer dataset, builds a baseline audience-persona classifier, matches a product query to creator niches, and ranks candidates for Direct Sales or Mass Brand Awareness.

Run the cells from top to bottom. Read each section note before running its code. Any generated reach, view, buyer score, or price is a prototype estimate unless it comes directly from a measured source.

## 0. Setup, dataset, and project scope

The provided CSV is expected in the same folder as this notebook. The project also uses an `outputs/` folder for recommendation tables and a chart.

The dataset has no creator bios, raw comments, measured reach/views, or campaign conversions. TF-IDF therefore matches a product query to a manually written niche vocabulary. Reach, views, and sponsorship costs use explicit demo assumptions.

The supplied persona labels align exactly with CVI bands. Treat the classifier as learning those dataset labels, not as a validated predictor of purchases.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("influencer_buyer_intent_dataset.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH.resolve())

## Optional: create a separate synthetic dataset

This cell recreates a simple synthetic schema for experimentation. It writes to `synthetic_influencer_buyer_intent_dataset.csv` so it will not overwrite the supplied CSV. The persona labels below are assigned directly from CVI thresholds, which intentionally demonstrates why target leakage can happen.

In [ ]:
np.random.seed(42)
n_samples = 1000

synthetic_ids = [f"INF_{i:04d}" for i in range(1, n_samples + 1)]
platforms = np.random.choice(
    ["Instagram", "YouTube", "TikTok"], size=n_samples, p=[0.45, 0.35, 0.20]
)
niches = np.random.choice(
    ["Tech", "Beauty & Skincare", "Fitness", "Finance", "Gaming", "Travel"],
    size=n_samples,
)
tiers = np.random.choice(
    ["Nano", "Micro", "Macro", "Mega"], size=n_samples, p=[0.40, 0.40, 0.15, 0.05]
)
followers = np.select(
    [tiers == "Nano", tiers == "Micro", tiers == "Macro", tiers == "Mega"],
    [
        np.random.randint(1_000, 10_000, n_samples),
        np.random.randint(10_000, 100_000, n_samples),
        np.random.randint(100_000, 1_000_000, n_samples),
        np.random.randint(1_000_000, 10_000_000, n_samples),
    ],
)

synthetic_df = pd.DataFrame({
    "Influencer_ID": synthetic_ids,
    "Platform": platforms,
    "Niche": niches,
    "Tier": tiers,
    "Followers": followers,
    "Engagement_Rate": np.round(np.random.uniform(0.8, 8.5, n_samples), 2),
    "Save_Share_Rate": np.round(np.random.uniform(0.2, 4.0, n_samples), 2),
    "Comment_Intent_Score": np.round(np.random.uniform(-0.5, 0.9, n_samples), 2),
    "Bot_Risk_Index": np.round(np.random.uniform(0.01, 0.35, n_samples), 2),
})
synthetic_df["CVI"] = np.round(
    synthetic_df["Save_Share_Rate"] * 0.4
    + (synthetic_df["Comment_Intent_Score"] + 0.5) * 4 * 0.4
    - synthetic_df["Bot_Risk_Index"] * 2,
    2,
)
synthetic_df["Audience_Persona"] = np.where(
    synthetic_df["CVI"] > 3.0,
    "High-Intent Buyers",
    np.where(synthetic_df["CVI"] > 1.5, "Casual Browsers", "Passive Watchers"),
)

synthetic_path = Path("synthetic_influencer_buyer_intent_dataset.csv")
synthetic_df.to_csv(synthetic_path, index=False)
print("Synthetic rows and columns:", synthetic_df.shape)
print("Saved separately to:", synthetic_path.resolve())

## 1. Load the project dataset

Load the supplied CSV into `df`. The shape is `(rows, columns)`. `info()` shows data types and non-empty counts; `head()` previews records.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Rows and columns:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
display(df.head())

## 2. Data quality checks

Check missing values, duplicate IDs, data types, and numeric ranges before modeling. A column like `Followers` should be numeric; persona and category fields should be text labels.

In [ ]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing"))
print("Duplicate influencer IDs:", df["Influencer_ID"].duplicated().sum())
print("Data types:")
print(df.dtypes)

numeric_cols = [
    "Followers", "Engagement_Rate", "Save_Share_Rate",
    "Comment_Intent_Score", "Bot_Risk_Index", "CVI",
]
display(df[numeric_cols].describe().T.round(2))
print("Persona percentages:")
print((df["Audience_Persona"].value_counts(normalize=True) * 100).round(1))

## 3. Compare audience personas

`groupby` splits creators by persona. `agg` computes counts and average metrics. This is descriptive analysis: an average difference does not show that a feature causes purchases.

In [ ]:
persona_summary = (
    df.groupby("Audience_Persona")
      .agg(
          creator_count=("Influencer_ID", "count"),
          avg_followers=("Followers", "mean"),
          avg_engagement_rate=("Engagement_Rate", "mean"),
          avg_save_share_rate=("Save_Share_Rate", "mean"),
          avg_comment_intent=("Comment_Intent_Score", "mean"),
          avg_bot_risk=("Bot_Risk_Index", "mean"),
          avg_cvi=("CVI", "mean"),
      )
      .round(2)
)
display(persona_summary)

## 4. Check CVI and label leakage

CVI means **Commercial Viability Index**. In this dataset, persona labels appear to be assigned from CVI ranges. If a classifier predicts persona, CVI must be excluded; otherwise it would have direct access to the rule that created the label.

In [ ]:
print("Overall CVI summary:")
print(df["CVI"].describe().round(2))
print("CVI summary by persona:")
display(
    df.groupby("Audience_Persona")["CVI"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

cvi_band = pd.cut(
    df["CVI"],
    bins=[-float("inf"), 1.5, 3.0, float("inf")],
    labels=["Low CVI", "Mid CVI", "High CVI"],
)
display(pd.crosstab(cvi_band, df["Audience_Persona"], normalize="index").round(2))

## 5. Define the model inputs and target

`X` contains the inputs given to the classifier; `y` is the persona label to predict. Drop `Influencer_ID` because it is only a key, and drop `CVI` because it appears to define `Audience_Persona`.

In [ ]:
target_col = "Audience_Persona"
excluded_cols = ["Influencer_ID", target_col, "CVI"]
X = df.drop(columns=excluded_cols)
y = df[target_col]

print("Feature matrix:", X.shape)
print("Target:", y.shape)
print("Features:", X.columns.tolist())

## 6. Split the data and train a baseline model

Keep a test split for final evaluation. Split part of the remaining training data into a fit set and validation set for threshold exploration. `stratify` keeps the rare High-Intent Buyer class represented in each split.

`ColumnTransformer` sends text categories to `OneHotEncoder` and numeric fields to `StandardScaler`. The pipeline fits these transformations on training data and applies them consistently later.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, random_state=43, stratify=y_train
)

categorical_features = ["Platform", "Niche", "Tier"]
numeric_features = [
    "Followers", "Engagement_Rate", "Save_Share_Rate",
    "Comment_Intent_Score", "Bot_Risk_Index",
]
preprocessor = ColumnTransformer(
    transformers=[
        ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numbers", StandardScaler(), numeric_features),
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ]
)
model.fit(X_fit, y_fit)
print("Fit rows:", len(X_fit), "| Validation rows:", len(X_valid), "| Test rows:", len(X_test))

## 7. Evaluate by persona

Inspect balanced accuracy and per-class precision/recall. Accuracy alone can look high when one class is much more common than another. The High-Intent Buyer validation support is small, so its metrics have high uncertainty.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

for split_name, X_part, y_part in [("Training", X_fit, y_fit), ("Validation", X_valid, y_valid)]:
    predictions = model.predict(X_part)
    print(f"\\n{split_name}")
    print("Balanced accuracy:", round(balanced_accuracy_score(y_part, predictions), 3))
    print(classification_report(y_part, predictions, zero_division=0))

valid_predictions = model.predict(X_valid)
cm = confusion_matrix(y_valid, valid_predictions, labels=model.classes_)
display(pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in model.classes_],
    columns=[f"Predicted: {label}" for label in model.classes_],
))

## 8. Choose a buyer-score cutoff on validation data

`predict_proba` returns the model's class scores. Higher cutoffs usually select fewer creators, often increasing precision while lowering recall. The score is not calibrated as a real purchase probability.

In [ ]:
from sklearn.metrics import precision_score, recall_score

buyer_column = list(model.classes_).index("High-Intent Buyers")
valid_probabilities = model.predict_proba(X_valid)[:, buyer_column]
actual_buyer_valid = y_valid.eq("High-Intent Buyers")
threshold_results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    selected = valid_probabilities >= threshold
    threshold_results.append({
        "threshold": threshold,
        "creators_selected": int(selected.sum()),
        "precision": precision_score(actual_buyer_valid, selected, zero_division=0),
        "recall": recall_score(actual_buyer_valid, selected, zero_division=0),
    })

display(pd.DataFrame(threshold_results).round(2))

## 9. Match product text to creator niches

The creator side uses a manually written niche vocabulary because bios are unavailable. The cosine score measures text similarity, not the probability of a product sale.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

niche_descriptions = {
    "Tech": "technology electronics gadgets audio headphones earbuds wireless devices",
    "Beauty & Skincare": "beauty skincare cosmetics makeup haircare",
    "Finance": "finance investing banking money personal finance",
    "Gaming": "video games esports gaming streams consoles",
    "Travel": "travel destinations tourism hotels trip planning",
    "Fitness": "fitness workouts exercise training wellness",
}
product_query = "Wireless Earbuds, Tech, wireless audio, headphones, consumer electronics"
creator_text = df["Niche"].map(niche_descriptions).fillna(df["Niche"].astype(str))
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
text_matrix = vectorizer.fit_transform(creator_text.tolist() + [product_query])
df["Semantic_Match_Score"] = cosine_similarity(text_matrix[-1], text_matrix[:-1]).ravel()
display(
    df[["Niche", "Semantic_Match_Score"]]
      .drop_duplicates("Niche")
      .sort_values("Semantic_Match_Score", ascending=False)
)

## 10. Out-of-fold scores for the full creator catalog

Out-of-fold predictions score every creator with a fold model that did not train on that row. Map scores and predicted personas by creator ID after any table merges; DataFrame merges can reset row indexes.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probabilities = cross_val_predict(model, X, y, cv=cv, method="predict_proba")
oof_predicted_personas = cross_val_predict(model, X, y, cv=cv, method="predict")
buyer_column = list(model.classes_).index("High-Intent Buyers")
oof_buyer_scores = oof_probabilities[:, buyer_column]

df_scored = df.copy()
df_scored["Buyer class score (%)"] = oof_buyer_scores * 100
df_scored["Predicted Audience Persona"] = oof_predicted_personas
print("Out-of-fold balanced accuracy:", round(balanced_accuracy_score(y, oof_predicted_personas), 3))
print(classification_report(y, oof_predicted_personas, zero_division=0))

## 11. Estimate reach, views, and sponsorship cost

Measured reach and views are missing. The following platform multipliers are placeholders for a portfolio demo. Replace them with actual post metrics. Pricing uses niche CPM assumptions, platform price factors, engagement adjustment, and a $75 minimum.

In [ ]:
platform_reach_rate = {"Instagram": 0.20, "YouTube": 0.25, "TikTok": 0.30}
platform_views_per_reach = {"Instagram": 1.35, "YouTube": 1.75, "TikTok": 1.60}
niche_cpm = {"Tech": 18, "Finance": 24, "Fitness": 14, "Beauty & Skincare": 15, "Gaming": 12, "Travel": 16}
platform_price_factor = {"Instagram": 1.00, "YouTube": 1.18, "TikTok": 0.88}
reference_engagement = df_scored["Engagement_Rate"].median()

candidates = df_scored[df_scored["Semantic_Match_Score"] > 0].copy()
candidates["Estimated Reach per Post"] = (
    candidates["Followers"] * candidates["Platform"].map(platform_reach_rate)
)
candidates["Estimated Views per Post"] = (
    candidates["Estimated Reach per Post"] * candidates["Platform"].map(platform_views_per_reach)
)
candidates["Engagement_Factor"] = (
    0.8 + 0.2 * candidates["Engagement_Rate"] / reference_engagement
).clip(0.75, 1.35)
candidates["Estimated Sponsorship Cost ($)"] = (
    candidates["Estimated Reach per Post"] / 1000
    * candidates["Niche"].map(niche_cpm).fillna(12)
    * candidates["Platform"].map(platform_price_factor).fillna(1.0)
    * candidates["Engagement_Factor"]
    + candidates["Followers"] / 1000 * 0.35
).clip(lower=75).round(2)
candidates["Estimated Engagements"] = (
    candidates["Followers"] * candidates["Engagement_Rate"] / 100
).clip(lower=1)
candidates["CPE ($)"] = (
    candidates["Estimated Sponsorship Cost ($)"] / candidates["Estimated Engagements"]
).round(2)
candidates["CPM ($)"] = (
    candidates["Estimated Sponsorship Cost ($)"] / candidates["Estimated Reach per Post"] * 1000
).round(2)

## 12. Build direct-sales and awareness rankings

Direct sales emphasizes the out-of-fold buyer-class score, product-niche match, and low bot risk. Awareness emphasizes views, reach, and low bot risk. The $3,000 budget is a demo campaign constraint.

In [ ]:
budget_usd = 3000
buyer_threshold = 80

candidates["Direct sales score (%)"] = 100 * (
    0.55 * candidates["Buyer class score (%)"] / 100
    + 0.25 * candidates["Semantic_Match_Score"]
    + 0.20 * (1 - candidates["Bot_Risk_Index"])
)
candidates["Reach Rank"] = candidates["Estimated Reach per Post"].rank(pct=True)
candidates["Views Rank"] = candidates["Estimated Views per Post"].rank(pct=True)
candidates["Awareness Score"] = (
    0.45 * candidates["Views Rank"]
    + 0.35 * candidates["Reach Rank"]
    + 0.20 * (1 - candidates["Bot_Risk_Index"])
)

direct_sales_affordable = candidates[
    (candidates["Buyer class score (%)"] >= buyer_threshold)
    & (candidates["Estimated Sponsorship Cost ($)"] <= budget_usd)
].sort_values("Direct sales score (%)", ascending=False).copy()

affordable_awareness = candidates[
    candidates["Estimated Sponsorship Cost ($)"] <= budget_usd
].sort_values("Awareness Score", ascending=False).copy()

print("Affordable direct-sales candidates:", len(direct_sales_affordable))
print("Affordable awareness candidates:", len(affordable_awareness))

## 13. Present the top-K results

The persona shown here is the out-of-fold prediction. Keep the source persona separately for evaluation; do not present it as a model prediction.

In [ ]:
direct_top10 = direct_sales_affordable.head(10).copy()
direct_top10.insert(0, "Rank", range(1, len(direct_top10) + 1))
direct_top10["Match Score (%)"] = direct_top10["Direct sales score (%)"].round(1)
direct_top10["Marketing Suitability"] = np.where(
    (direct_top10["Predicted Audience Persona"] == "High-Intent Buyers")
    & (direct_top10["Bot_Risk_Index"] <= 0.20),
    "Highly Recommended for Direct Sales",
    "Review Before Direct Sales",
)

display(direct_top10[[
    "Rank", "Influencer_ID", "Platform", "Niche", "Followers",
    "Estimated Reach per Post", "Engagement_Rate", "Bot_Risk_Index",
    "Buyer class score (%)", "Predicted Audience Persona", "Match Score (%)",
    "Estimated Sponsorship Cost ($)", "CPE ($)", "CPM ($)", "Marketing Suitability",
]].round(2))

awareness_top10 = affordable_awareness.head(10).copy()
awareness_top10.insert(0, "Rank", range(1, len(awareness_top10) + 1))
awareness_top10["Product-Niche Match (%)"] = (
    awareness_top10["Semantic_Match_Score"] * 100
).round(1)
awareness_top10["Match Score (%)"] = (awareness_top10["Awareness Score"] * 100).round(1)
awareness_top10["Marketing Suitability"] = "Best for Awareness Only"

display(awareness_top10[[
    "Rank", "Influencer_ID", "Platform", "Niche", "Followers",
    "Estimated Reach per Post", "Estimated Views per Post", "Bot_Risk_Index",
    "Audience_Persona", "Product-Niche Match (%)", "Match Score (%)",
    "Estimated Sponsorship Cost ($)", "Marketing Suitability",
]].round(2))

## 14. Save outputs for the dashboard

`app.py` reads these exact filenames. The app displays precomputed recommendation CSVs; rerun this notebook to refresh them.

In [ ]:
direct_top10.to_csv(OUTPUT_DIR / "direct_sales_recommendation.csv", index=False)
awareness_top10.to_csv(OUTPUT_DIR / "awareness_recommendations.csv", index=False)
print("Saved recommendation CSVs under:", OUTPUT_DIR.resolve())

## 15. Buyer vs. Watcher visualization

This chart compares mean signals by source persona. Feature scales differ, so interpret each panel on its own scale.

In [ ]:
persona_order = ["High-Intent Buyers", "Casual Browsers", "Passive Watchers"]
metrics = ["Save_Share_Rate", "Comment_Intent_Score", "Bot_Risk_Index", "CVI"]
persona_means = df.groupby("Audience_Persona")[metrics].mean().reindex(persona_order)
titles = ["Average Save/Share Rate", "Average Comment Intent", "Average Bot Risk", "Average CVI"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric, title in zip(axes.flat, metrics, titles):
    persona_means[metric].plot(kind="bar", ax=ax, color=["#2a9d8f", "#e9c46a", "#e76f51"])
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "buyer_vs_watcher_metrics.png", dpi=200, bbox_inches="tight")
plt.show()

## 16. Run the Streamlit dashboard

Install dependencies from `requirements.txt`, then run this command from the project folder:

```bash
python -m streamlit run app.py
```

## Final interview notes

This is a hybrid recommendation prototype using Logistic Regression, TF-IDF niche matching, objective-specific ranking, bot-risk signals, and rule-based pricing. The dataset labels appear CVI-derived; reach, views, and fees are proxies; and no purchase outcomes are available. State these limits clearly in a portfolio or interview.